In [5]:
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score

import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [6]:
X = np.load("../processed/X.npy")
y = np.load("../processed/y.npy")

print(X.shape)
print(y.shape)

(4635, 7, 297)
(4635,)


In [7]:
skf = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

In [8]:
acc_scores = []
prec_scores = []
rec_scores = []

In [9]:
fold = 1

for train_idx, test_idx in skf.split(X, y):

    print(f"\n===== Fold {fold} =====")

    X_train = X[train_idx]
    X_test = X[test_idx]

    y_train = y[train_idx]
    y_test = y[test_idx]

    # Standardization

    scaler = StandardScaler()

    X_train_flat = X_train.reshape(-1, 297)
    X_test_flat = X_test.reshape(-1, 297)

    X_train_flat = scaler.fit_transform(
        X_train_flat
    )

    X_test_flat = scaler.transform(
        X_test_flat
    )

    X_train = X_train_flat.reshape(
        X_train.shape
    )

    X_test = X_test_flat.reshape(
        X_test.shape
    )

    # Model

    inputs = Input(shape=(7,297))

    x = LSTM(
        256,
        return_sequences=True
    )(inputs)

    x = Dropout(0.2)(x)

    x = LSTM(
        256,
        return_sequences=True
    )(x)

    x = Dropout(0.1)(x)

    x = LSTM(
        256,
        return_sequences=False
    )(x)

    x = Dropout(0.2)(x)

    outputs = Dense(
        1,
        activation="sigmoid"
    )(x)

    model = Model(inputs, outputs)

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )

    model.fit(
        X_train,
        y_train,
        validation_split=0.1,
        epochs=100,
        batch_size=32,
        callbacks=[early_stop],
        verbose=0
    )

    pred = model.predict(
        X_test,
        verbose=0
    )

    pred = (pred > 0.5).astype(int)

    acc = accuracy_score(
        y_test,
        pred
    )

    prec = precision_score(
        y_test,
        pred
    )

    rec = recall_score(
        y_test,
        pred
    )

    acc_scores.append(acc)
    prec_scores.append(prec)
    rec_scores.append(rec)

    print("Accuracy :", round(acc,4))
    print("Precision:", round(prec,4))
    print("Recall   :", round(rec,4))

    fold += 1


===== Fold 1 =====
Accuracy : 0.7026
Precision: 0.715
Recall   : 0.6652

===== Fold 2 =====
Accuracy : 0.6853
Precision: 0.681
Recall   : 0.687

===== Fold 3 =====
Accuracy : 0.722
Precision: 0.7205
Recall   : 0.7174

===== Fold 4 =====
Accuracy : 0.6983
Precision: 0.6907
Recall   : 0.7087

===== Fold 5 =====
Accuracy : 0.6918
Precision: 0.6719
Recall   : 0.7391

===== Fold 6 =====
Accuracy : 0.6911
Precision: 0.69
Recall   : 0.687

===== Fold 7 =====
Accuracy : 0.7019
Precision: 0.6901
Recall   : 0.7261

===== Fold 8 =====
Accuracy : 0.7646
Precision: 0.7665
Recall   : 0.7565

===== Fold 9 =====
Accuracy : 0.6242
Precision: 0.6291
Recall   : 0.5852

===== Fold 10 =====
Accuracy : 0.6955
Precision: 0.688
Recall   : 0.7031


In [10]:
print("\n========== FINAL ==========")

print(
    "Mean Accuracy:",
    np.mean(acc_scores)
)

print(
    "Mean Precision:",
    np.mean(prec_scores)
)

print(
    "Mean Recall:",
    np.mean(rec_scores)
)


========== FINAL ==========
Mean Accuracy: 0.6977321814254859
Mean Precision: 0.6942827465798103
Mean Recall: 0.6975166128726029
